# 02 — Dataset Check

Verifies the 70 / 15 / 15 stratified split and shows class-distribution bar charts.

**Run after** `create_splits()` has been called (or `train.py` has been run once, which auto-calls it).  
CSVs expected at `data/train.csv`, `data/val.csv`, `data/test.csv`, `data/classes.csv`.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
DATA_DIR = Path('../data')

train_df   = pd.read_csv(DATA_DIR / 'train.csv')
val_df     = pd.read_csv(DATA_DIR / 'val.csv')
test_df    = pd.read_csv(DATA_DIR / 'test.csv')
classes_df = pd.read_csv(DATA_DIR / 'classes.csv')

class_names = classes_df['family'].tolist()
idx_to_name = dict(enumerate(class_names))

print(f"Train : {len(train_df):>5} samples")
print(f"Val   : {len(val_df):>5} samples")
print(f"Test  : {len(test_df):>5} samples")
print(f"Total : {len(train_df) + len(val_df) + len(test_df):>5} samples")
print(f"Classes ({len(class_names)}): {class_names}")

In [ ]:
def count_per_class(df, n_classes, idx_to_name):
    counts = df['label'].value_counts().sort_index()
    return [counts.get(i, 0) for i in range(n_classes)]

n = len(class_names)
train_counts = count_per_class(train_df, n, idx_to_name)
val_counts   = count_per_class(val_df,   n, idx_to_name)
test_counts  = count_per_class(test_df,  n, idx_to_name)

summary = pd.DataFrame({
    'Family': class_names,
    'Train':  train_counts,
    'Val':    val_counts,
    'Test':   test_counts,
})
summary['Total'] = summary['Train'] + summary['Val'] + summary['Test']
display(summary)

In [ ]:
x = np.arange(n)
width = 0.28

fig, ax = plt.subplots(figsize=(max(10, n * 1.5), 5))
bars_train = ax.bar(x - width, train_counts, width, label='Train', color='steelblue')
bars_val   = ax.bar(x,         val_counts,   width, label='Val',   color='orange')
bars_test  = ax.bar(x + width, test_counts,  width, label='Test',  color='seagreen')

ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Sample count')
ax.set_title('Class distribution across splits (70 / 15 / 15)')
ax.legend()
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('../outputs/split_distribution.png', dpi=150)
plt.show()
print('Saved to outputs/split_distribution.png')

## Class imbalance check

Computes inverse-frequency class weights — these are fed directly to `CrossEntropyLoss` in `train.py`.

In [ ]:
import numpy as np

counts = np.array(train_counts, dtype=float)
weights = 1.0 / (counts + 1e-6)
weights = weights / weights.sum() * n

print("Class weights (inverse frequency, used in CrossEntropyLoss):")
for name, w in zip(class_names, weights):
    print(f"  {name:<25} {w:.4f}")

max_ratio = counts.max() / (counts.min() + 1e-6)
print(f"\nImbalance ratio (max/min class): {max_ratio:.1f}x")
if max_ratio > 10:
    print("  WARNING: severe imbalance — consider oversampling the minority class.")
elif max_ratio > 3:
    print("  Moderate imbalance — class weights should handle this.")
else:
    print("  Imbalance is mild — class weights are a safe precaution.")